In [1]:
import pandas as pd

In [2]:
df = pd.read_excel("Данные для тестового задания.xlsx")

In [3]:
mau = df["user_id"].nunique()

print("MAU =", mau)

MAU = 7639


In [5]:
average_dau = dau.mean()

print("Средний DAU =", average_dau)

Средний DAU = 560.4666666666667


In [7]:
print(df.columns.tolist())

['date', 'user_id', 'view_adverts']


In [8]:
print(df.head())

        date                               user_id  view_adverts
0 2023-11-11  8c020470-8461-11ed-83d0-552e8cc749d6            13
1 2023-11-18  5875f070-7b92-11ee-a6fb-8b298e83f4f7            14
2 2023-11-29  3c2d27c0-4fd6-11eb-b89f-2ffb31b67dd6            21
3 2023-11-29  234a96d0-ad16-11ed-a2e6-793ddfeeba1f            23
4 2023-11-29  4d07c180-644f-11eb-879c-b7c02edf4f37            12


In [9]:
df["date"] = pd.to_datetime(df["date"])

# Пользователи, которые были в продукте 1 ноября 2023
users_nov_1 = set(
    df[df["date"].dt.date == pd.Timestamp("2023-11-01").date()]["user_id"]
)

# Те же пользователи, которые вернулись 2 ноября 2023
users_nov_2 = set(
    df[df["date"].dt.date == pd.Timestamp("2023-11-02").date()]["user_id"]
)

# Количество вернувшихся пользователей
returned_users = users_nov_1 & users_nov_2

# D1 Retention
retention_d1 = len(returned_users) / len(users_nov_1) * 100

print("Пользователей 1 ноября:", len(users_nov_1))
print("Вернулось 2 ноября:", len(returned_users))
print("D1 Retention:", retention_d1, "%")

Пользователей 1 ноября: 623
Вернулось 2 ноября: 166
D1 Retention: 26.64526484751204 %


In [10]:
total_users = df["user_id"].nunique()

# Количество уникальных пользователей, посмотревших хотя бы одно объявление
users_with_ad = df[df["view_adverts"] > 0]["user_id"].nunique()

# Конверсия
conversion = users_with_ad / total_users * 100

print("Всего пользователей:", total_users)
print("Пользователей, посмотревших объявления:", users_with_ad)
print("Конверсия:", conversion, "%")

Всего пользователей: 7639
Пользователей, посмотревших объявления: 3538
Конверсия: 46.31496269145176 %


In [11]:
total_adverts = df["view_adverts"].sum()

# Количество уникальных пользователей
total_users = df["user_id"].nunique()

# Среднее количество объявлений на пользователя
avg_adverts = total_adverts / total_users

print("Среднее количество объявлений на пользователя:", avg_adverts)

Среднее количество объявлений на пользователя: 2.8687000916350307


In [15]:
df_ab = pd.read_excel(
    "Данные для тестового задания.xlsx",
    sheet_name="Данные АБ тестов"
)

In [16]:
print(df_ab.columns.tolist())

['experiment_num', 'experiment_group', 'user_id', 'revenue']


In [17]:
print(df_ab.groupby(["experiment_num", "experiment_group"])["user_id"].count())

experiment_num  experiment_group
1               control             465
                test                480
2               control             465
                test                480
3               control             465
                test                480
Name: user_id, dtype: int64


In [18]:
from scipy.stats import ttest_ind

for experiment in sorted(df_ab["experiment_num"].unique()):

    exp = df_ab[df_ab["experiment_num"] == experiment]

    control = exp[exp["experiment_group"] == "control"]["revenue"]
    test = exp[exp["experiment_group"] == "test"]["revenue"]

    # ARPU
    arpu_control = control.mean()
    arpu_test = test.mean()

    # t-test
    statistic, p_value = ttest_ind(
        control,
        test,
        equal_var=False
    )

    # Относительное изменение ARPU
    relative_change = (arpu_test / arpu_control - 1) * 100

    print(f"Эксперимент {experiment}")
    print(f"ARPU control: {arpu_control:.2f}")
    print(f"ARPU test: {arpu_test:.2f}")
    print(f"Изменение ARPU: {relative_change:.2f}%")
    print(f"p-value: {p_value:.4f}")
    print("-" * 40)

Эксперимент 1
ARPU control: 722.46
ARPU test: 665.74
Изменение ARPU: -7.85%
p-value: 0.6890
----------------------------------------
Эксперимент 2
ARPU control: 704.65
ARPU test: 332.93
Изменение ARPU: -52.75%
p-value: 0.0011
----------------------------------------
Эксперимент 3
ARPU control: 663.21
ARPU test: 998.67
Изменение ARPU: 50.58%
p-value: 0.0603
----------------------------------------


In [21]:
df_ab.head()

,experiment_num,experiment_group,user_id,revenue
0,1,test,38456,520
1,1,control,13125924,806
2,1,control,9761984,0
3,1,test,11387012,208
4,1,test,18319648,104


In [25]:
print(df_ab.shape)
print(df_ab.head())
print(df_ab["experiment_num"].value_counts())

(2835, 4)
   experiment_num experiment_group   user_id  revenue
0               1             test     38456      520
1               1          control  13125924      806
2               1          control   9761984        0
3               1             test  11387012      208
4               1             test  18319648      104
experiment_num
1    945
2    945
3    945
Name: count, dtype: int64


In [26]:
print(df_ab.groupby("experiment_num")["revenue"].agg(["sum", "count"]))

                   sum  count
experiment_num               
1               655499    945
2               487470    945
3               787752    945


In [27]:
print(df_ab.groupby("experiment_num")["user_id"].nunique())

experiment_num
1    945
2    945
3    945
Name: user_id, dtype: int64


In [28]:
for exp in sorted(df_ab["experiment_num"].unique()):
    data = df_ab[df_ab["experiment_num"] == exp]
    
    arpu = data["revenue"].sum() / data["user_id"].nunique()
    
    print(f"Эксперимент {exp}: {arpu:.2f}")

Эксперимент 1: 693.65
Эксперимент 2: 515.84
Эксперимент 3: 833.60


In [30]:
df_l = pd.read_excel(
    "Данные для тестового задания.xlsx",
    sheet_name="Листеры"
)

In [31]:
df_l.head()


,user_id,date,cnt_adverts,age,cnt_contacts,revenue
0,100,2022-01-01,6,21,119,53
1,100,2022-01-02,2,21,200,18
2,100,2022-01-03,6,21,193,42
3,100,2022-01-04,2,21,143,38
4,100,2022-01-05,2,21,190,40


In [32]:
total_revenue = df_l["revenue"].sum()

# Количество уникальных пользователей
unique_users = df_l["user_id"].nunique()

# Средний доход на пользователя
arpu = total_revenue / unique_users

print("Общая выручка:", total_revenue)
print("Уникальных пользователей:", unique_users)
print("Средний доход на пользователя:", round(arpu, 2))

Общая выручка: 4851
Уникальных пользователей: 31
Средний доход на пользователя: 156.48


In [33]:
median_age = df_l["age"].median()

print("Медиана возраста:", median_age)

Медиана возраста: 28.0
